## imports


In [20]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import sys, json, random, asyncio
import torch
import torch.nn.functional as F
import nest_asyncio
nest_asyncio.apply()
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType
from data.load_data          import DataLoader, TrainingQuery
from src.state_extractor import StateExtractor
from src.courtroom_env   import CourtroomEnv
from src.llm_judge   import LLMJudge
import yaml

ROOT = Path().absolute().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device:  {device}')
print(f'PyTorch: {torch.__version__}')
print(f'ROOT:    {ROOT}')
print('All modules imported ✅')

Device:  mps
PyTorch: 2.10.0
ROOT:    /Users/ajit/Desktop
All modules imported ✅


In [10]:

with open(ROOT /'Role_Policy'/'configs'/ 'training.yaml') as f:
    cfg = yaml.safe_load(f)

# DAPO hyperparameters
GROUP_SIZE          = cfg['dapo']['group_size']           # 8
EPSILON_HIGH        = cfg['dapo']['epsilon_high']         # 0.2
EPSILON_LOW         = cfg['dapo']['epsilon_low']          # 0.1
ADVANTAGE_THRESHOLD = cfg['dapo']['advantage_threshold']  # 0.01
LEARNING_RATE       = float(cfg['dapo']['learning_rate']) # 1e-5
MAX_ITERATIONS      = cfg['dapo']['max_iterations']       # 500
CHECKPOINT_EVERY    = cfg['dapo']['checkpoint_every']     # 50
BASE_MODEL          = cfg['dapo']['base_model']
MAX_NEW_TOKENS      = 256

print(f'Model:      {BASE_MODEL}')
print(f'Iterations: {MAX_ITERATIONS}')
print(f'G:          {GROUP_SIZE}')
print(f'LR:         {LEARNING_RATE}')
print(f'ε+/ε-:      {EPSILON_HIGH}/{EPSILON_LOW}')

Model:      Qwen/Qwen2.5-3B-Instruct
Iterations: 500
G:          8
LR:         1e-05
ε+/ε-:      0.2/0.1


In [11]:
loader = DataLoader()
splits = loader.load_all(
    squad_n        = cfg['training_data']['sources']['squad'],
    snli_n         = cfg['training_data']['sources']['snli'],
    contract_n     = cfg['training_data']['sources']['contractnli'],
    synthetic_path = str(ROOT / 'data' / 'synthetic' / 'queries.jsonl'),
)

train_queries = splits['train']
val_queries   = splits['val']

print(f'\ntrain : {len(train_queries)}')
print(f'val   : {len(val_queries)}')
print(f'test  : {len(splits["test"])}')

[DataLoader] Loading SQuAD (400 examples)...
[DataLoader] Loaded 400 SQuAD queries
[DataLoader] Loading SNLI (200 examples)...
[DataLoader] Loaded 200 SNLI queries
[DataLoader] Loading ContractNLI (100 examples)...
[DataLoader] Loaded 100 ContractNLI queries
[DataLoader] Loading synthetic queries from /Users/ajit/Desktop/data/synthetic/queries.jsonl...
[DataLoader] Not found: /Users/ajit/Desktop/data/synthetic/queries.jsonl. Run generate_synthetic.py first.

[DataLoader] Total queries loaded: 700
  train : 560
  val   : 70
  test  : 70

train : 560
val   : 70
test  : 70


In [13]:
# Cell 5 — Load Qwen 3B + LoRA
print(f'Loading {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype = torch.float16,
    device_map  = device,
)

lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ['q_proj', 'v_proj'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print('Model + optimizer ready ✅')

Loading Qwen/Qwen2.5-3B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:09<00:00, 43.69it/s]


trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
Model + optimizer ready ✅


In [14]:
# Cell 6 — Test single generation (inspect tensor shapes)
test_prompt = 'Assign roles: prosecutor, defense, judge_factual, judge_reasoning, judge_clarity'
inputs      = tokenizer(test_prompt, return_tensors='pt',
                        truncation=True, max_length=512).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens          = 50,
        temperature             = 0.8,
        do_sample               = True,
        pad_token_id            = tokenizer.eos_token_id,
        return_dict_in_generate = True,
        output_scores           = True,
    )

gen_ids = outputs.sequences[0][inputs['input_ids'].shape[1]:]
text    = tokenizer.decode(gen_ids, skip_special_tokens=True)
scores  = torch.stack(outputs.scores, dim=0)

print(f'Generated:      {text[:80]}')
print(f'gen_ids shape:  {gen_ids.shape}')
print(f'scores shape:   {scores.shape}')   # ← key diagnostic
print(f'scores.dim():   {scores.dim()}')

Generated:      , witness_1, witness_2
Prosecutor: I will be the prosecutor in this hypothetical
gen_ids shape:  torch.Size([50])
scores shape:   torch.Size([50, 1, 151936])
scores.dim():   3


In [27]:
# Cell 7 — Helper functions
extractor = StateExtractor()
env       = CourtroomEnv()
judge     = LLMJudge()

def generate_assignment(prompt: str):
    messages = [
        {
            "role": "system",
            "content": "You are a role assignment policy. You ONLY output valid JSON. Never explain. Never add text before or after the JSON."
        },
        {"role": "user", "content": prompt}
    ]
    text_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text_input, return_tensors='pt',
        truncation=True, max_length=768,
    ).to(device)

    # Step 1 — Generate tokens (no grad needed here)
    with torch.no_grad():
        outputs = model.generate(      # ← indented under with block
            **inputs,
            max_new_tokens          = 512,
            temperature             = 0.8,
            do_sample               = True,
            pad_token_id            = tokenizer.eos_token_id,
            return_dict_in_generate = True,
        )

    gen_ids = outputs.sequences[0][inputs['input_ids'].shape[1]:]
    text    = tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Step 2 — Recompute log_probs WITH grad enabled
    full_ids  = outputs.sequences                                        # [1, full_len]
    logits    = model(full_ids).logits                                   # [1, seq_len, vocab]
    logits    = logits[0, inputs['input_ids'].shape[1]-1:-1, :]         # [gen_len, vocab]
    log_probs = F.log_softmax(logits, dim=-1)                           # [gen_len, vocab]
    token_lps = log_probs[range(len(gen_ids)), gen_ids].sum()           # scalar with grad

    return text, token_lps


def parse_assignment(content: str):
    from src.courtroom_env import RoleAssignment
    import re
    try:
        # Try direct parse first
        raw = content.strip()
        
        # Strip markdown fences
        if '```' in raw:
            raw = re.search(r'```(?:json)?\s*([\s\S]*?)```', raw)
            raw = raw.group(1).strip() if raw else content
        
        # Find JSON object if there's surrounding text
        if not raw.startswith('{'):
            match = re.search(r'\{[\s\S]*\}', raw)
            raw   = match.group(0) if match else raw
        
        return RoleAssignment.from_dict(json.loads(raw))
    except Exception as e:
        print(f'  [parse_assignment failed] {e}')
        print(f'  raw output: {content[:150]}')
        return None


def compute_dapo_loss(log_probs, rewards):
    """DAPO group-normalized loss with decoupled clipping + dead sample filter."""
    if len(rewards) < 2:
        return None

    mean_r = sum(rewards) / len(rewards)
    var_r  = sum((r - mean_r)**2 for r in rewards) / len(rewards)
    std_r  = var_r**0.5 + 1e-8

    loss   = torch.tensor(0.0, device=device)
    n_used = 0

    for lp, r in zip(log_probs, rewards):
        a = (r - mean_r) / std_r
        if abs(a) < ADVANTAGE_THRESHOLD:
            continue
        a = min(a, EPSILON_HIGH) if a > 0 else max(a, -EPSILON_LOW)
        loss   = loss + (-a * lp)
        n_used += 1

    return loss / n_used if n_used > 0 else None


print('Helper functions defined ✅')

Helper functions defined ✅


In [ ]:
# Cell 8 — DRY RUN: one iteration manually
# Verify full pipeline before running the loop

query  = train_queries[0]
state  = extractor.extract(query.query, query.doc_type)
prompt = state.to_prompt()

print(f'Query:  {query.query[:80]}')
print(f'Source: {query.source}\n')

# Generate one assignment
text, lp = generate_assignment(prompt)
print(f'Generated assignment:\n{text[:300]}')
print(f'\nlog_prob: {lp.item():.4f}')

# Parse it
assignment = parse_assignment(text)
print(f'\nParsed: {"✅" if assignment else "❌ failed"}')

Query:  Does this hypothesis follow from the evidence? 'The people are eating omelettes.
Source: snli



In [23]:
# Cell 9 — DRY RUN: courtroom + reward
# Only run if Cell 8 parsed assignment successfully

if assignment:
    result = await env.run(query.query, query.evidence, assignment)
    reward = await judge.score(result, query.ground_truth)

    print(f'Prosecution: {result.prosecution_argument[:200]}')
    print(f'\nDefense:     {result.defense_argument[:200]}')
    print(f'\nVerdict:     {result.verdict}')
    print(f'Reward:      {reward}')
else:
    print('Skip — assignment parse failed in Cell 8')

Prosecution: [Model error: Client error '400 Bad Request' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400]

Defense:     [Model error: Client error '400 Bad Request' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400]

Verdict:     Confidence: 0.50. Draw prevailed on: Does this hypothesis follow from the evidence? 'The people are eating omelettes.
Reward:      0.45


In [24]:
# Cell 10 — FULL TRAINING LOOP
# Change N_ITERATIONS to control how many steps to run
# Recommended: start with 10, then 50, then full 500

N_ITERATIONS = 10 

Path('checkpoints').mkdir(exist_ok=True)
training_log = []

for iteration in range(N_ITERATIONS):
    query  = random.choice(train_queries)
    state  = extractor.extract(query.query, query.doc_type)
    prompt = state.to_prompt()

    print(f'\n[{iteration+1}/{N_ITERATIONS}] {query.query[:55]}...')

    # Generate G rollouts
    log_probs   = []
    assignments = []
    for _ in range(GROUP_SIZE):
        text, lp   = generate_assignment(prompt)
        assignment = parse_assignment(text)
        log_probs.append(lp)
        assignments.append(assignment)

    # Run courtroom for valid assignments
    tasks     = []
    valid_lps = []
    for lp, a in zip(log_probs, assignments):
        if a is not None:
            tasks.append(env.run(query.query, query.evidence, a))
            valid_lps.append(lp)

    if not tasks:
        print('  [SKIP] No valid assignments')
        continue

    results = await asyncio.gather(*tasks, return_exceptions=True)
    valid_results, final_lps = [], []
    for r, lp in zip(results, valid_lps):
        if not isinstance(r, Exception):
            valid_results.append(r)
            final_lps.append(lp)

    if not valid_results:
        print('  [SKIP] All courtroom runs failed')
        continue

    # Score with LLM judge
    rewards = await judge.score_group(valid_results, query.ground_truth)
    mean_r  = sum(rewards) / len(rewards)
    print(f'  rewards: min={min(rewards):.3f}  mean={mean_r:.3f}  max={max(rewards):.3f}')

    # DAPO loss + update
    loss = compute_dapo_loss(final_lps, rewards)
    if loss is not None:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f'  loss: {loss.item():.4f}')
        training_log.append({
            'iter':       iteration + 1,
            'mean_reward': round(mean_r, 4),
            'loss':        round(loss.item(), 4),
        })
    else:
        print('  [SKIP] All dead samples')

    # Checkpoint
    if (iteration + 1) % CHECKPOINT_EVERY == 0:
        ckpt = f'checkpoints/iter_{iteration+1}'
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f'  [Checkpoint] → {ckpt}')

print('\n✅ Training complete')


[1/10] Does this contract clause apply? 'Video cameras that I ...
  rewards: min=0.450  mean=0.450  max=0.450
  [SKIP] All dead samples

[2/10] The Basilica of the Sacred heart at Notre Dame is besid...
  [parse_assignment failed] Unterminated string starting at: line 6 column 51 (char 1207)
  raw output: {
  "prosecutor":       {"model": "Claude", "behavior": "Approach the document with a neutral stance, focusing on summarizing key points and extractin
  [parse_assignment failed] Unterminated string starting at: line 6 column 51 (char 1192)
  raw output: {
  "prosecutor":       {"model": "Claude", "behavior": "Assess the credibility and relevance of the research paper, focusing on its potential impact 
  rewards: min=0.250  mean=0.250  max=0.250
  [SKIP] All dead samples

[3/10] Who filed a lawsuit over Survivor?...
  rewards: min=0.000  mean=0.000  max=0.000
  [SKIP] All dead samples

[4/10] Does this hypothesis follow from the evidence? 'Two adu...
  [parse_assignment failed] Unter

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# Cell 11 — Plot training curve


if training_log:
    iters   = [r['iter']        for r in training_log]
    rewards = [r['mean_reward'] for r in training_log]
    losses  = [r['loss']        for r in training_log]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(iters, rewards, color='steelblue', linewidth=2)
    axes[0].set_title('Mean Reward per Iteration')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Reward')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iters, losses, color='tomato', linewidth=2)
    axes[1].set_title('DAPO Loss per Iteration')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('DAPO Training Curves', fontsize=13, fontweight='bold')
    plt.tight_layout()

    Path('exports').mkdir(exist_ok=True)
    plt.savefig('exports/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → exports/training_curves.png')
else:
    print('No training log yet — run Cell 10 first')